# 📓 Notebook A — Baseline Evaluation & LoRA Rank Ablation

**Experiments:** E00 (baseline) · E01 – E04 (rank ablation)  
**Estimated runtime:** ~13 GPU-hours on a free Colab T4  
**Run order:** top-to-bottom without skipping any cell

---

## Research Overview

This notebook is part of a multi-notebook study investigating **catastrophic forgetting** in large language models (LLMs) during parameter-efficient fine-tuning via **Low-Rank Adaptation (LoRA)**. Specifically, we ask:

> *When a pre-trained LLM is fine-tuned on a narrow domain (medicine), does it forget general knowledge — and, if so, does the pattern of forgetting correlate with semantic proximity to the fine-tuning domain?*

### Hypotheses tested here

| ID | Hypothesis | What we measure |
|----|-----------|-----------------|
| **H2** | LoRA rank modulates the severity of forgetting (Biderman 2024 vs Steele 2026) | Mean MMLU accuracy drop across E01–E04 |
| **H1** | Forgetting correlates positively with semantic similarity to the training domain | Pearson r between cosine sim and Δaccuracy |
| **H3** | Proximal (medically adjacent) MMLU subjects forget more than distal ones | Independent t-test: proximal vs distal Δacc |

### Notebook structure at a glance

```
Steps 0–3   Setup (install, Drive, config, shared utilities)
E00         Baseline: eval all 57 MMLU subjects on the unmodified model
E01–E04     Rank ablation: fine-tune at r=4, 16, 64, 128 → re-eval → compute forgetting
```

All results are written to Google Drive **immediately** after each experiment — a Colab crash loses nothing.


In [1]:
# --- CELL 0: Environment setup — MUST be first, before any other import ---
import os
os.environ["CUDA_VISIBLE_DEVICES"]        = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]     = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_CUDA_WARMUP"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"]        = "1"   # surfaces real errors immediately
print("✅ Single-GPU enforced")

✅ Single-GPU enforced


https://www.kaggle.com/code/akankshanarula/e00-notebook

## Step 0 — Install dependencies
Run once per Colab session. *Runtime → Run all* will handle this.

In [2]:
# --- CELL 0: Install ---
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Qwen3.5 architecture (qwen3_5) is not in any PyPI release yet (as of Mar 2026).
# We must install transformers from the git main branch to get Qwen3_5ForCausalLM.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/huggingface/transformers.git"
])

pip("peft>=0.12.0", "trl>=0.10.0",
    "bitsandbytes>=0.43.0", "accelerate>=0.33.0",
    "datasets>=2.20.0", "sentence-transformers>=3.0.0",
    "scipy", "scikit-learn", "matplotlib", "seaborn")

print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
✅ Dependencies installed


## Step 1 — Mount Google Drive
All results are saved here IMMEDIATELY after each experiment.  
If Colab crashes you lose nothing.

In [9]:
# --- CELL 1: Kaggle paths (replaces Colab Drive mount) ---
import os

DRIVE_BASE = "/kaggle/working/lora_forgetting_research"
os.makedirs(f"{DRIVE_BASE}/results",    exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/adapters",   exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/mmlu_evals", exist_ok=True)

RESULTS_CSV  = f"{DRIVE_BASE}/results/all_results.csv"
BASELINE_CSV = f"{DRIVE_BASE}/results/E00_baseline.csv"

# Copy uploaded checkpoint files into working dir so Cell 9 can resume
import shutil
UPLOAD_DIR = "/kaggle/input/lora-results"   # ← your dataset name here
for fname in ["E00_checkpoint.csv", "E00_baseline.csv", "all_results.csv"]:
    src = f"{UPLOAD_DIR}/{fname}"
    dst = f"{DRIVE_BASE}/results/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f"✅ Restored {fname}")

print(f"✅ Paths ready. Working directory: {DRIVE_BASE}")

✅ Paths ready. Working directory: /kaggle/working/lora_forgetting_research


In [10]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN_HERE")

---
## 🧠 Background Theory — Catastrophic Forgetting & LoRA

### What is catastrophic forgetting?

When a neural network is trained sequentially on two tasks, learning task B can **overwrite the weights** used for task A — a phenomenon called *catastrophic forgetting* (McCloskey & Cohen, 1989; Kirkpatrick et al., 2017). In the era of LLMs, this manifests as: after fine-tuning on a narrow corpus (e.g. medical QA), the model degrades on unrelated general-knowledge benchmarks.

The severity and *topology* of forgetting matter. Does the model forget **uniformly**, or does it forget more in subjects that are semantically related to the fine-tuning domain? This study tests the latter (Hypotheses H1 & H3).

---

### What is LoRA and why does rank matter?

**LoRA (Hu et al., 2022)** is a parameter-efficient fine-tuning (PEFT) method. Instead of updating all model weights $W \in \mathbb{R}^{d 	imes k}$, it freezes $W$ and learns a low-rank update:

$$\Delta W = B A, \quad B \in \mathbb{R}^{d 	imes r},\ A \in \mathbb{R}^{r 	imes k}, \quad r \ll \min(d, k)$$

Only $B$ and $A$ are trained. The adapter is applied as:

$$h = W_0 x +
rac{lpha}{r} \Delta W x$$

where $lpha$ is a scaling hyperparameter (commonly set to $2r$).

**Key insight about rank $r$:**
- Small $r$ (e.g. 4) → few free parameters → limited representational capacity → possibly less overwriting of general knowledge
- Large $r$ (e.g. 128) → many free parameters → higher capacity → potentially more forgetting of pre-trained knowledge

Whether rank *actually* modulates forgetting is empirically contested — **Biderman et al. (2024)** claim higher rank causes more forgetting, while **Steele (2026)** argues rank has no significant effect. **This notebook generates the data to resolve that contradiction (H2).**

---

### 4-bit QLoRA — fitting an 8B model on a free T4

A standard 8B parameter model in fp16 requires ~16 GB VRAM. The free Colab T4 has only ~15 GB. We use **QLoRA** (Dettmers et al., 2023):

1. **NF4 quantization** — weights stored in 4-bit Normal Float format (optimal for normally-distributed neural network weights), reducing memory to ~5.5 GB
2. **Double quantization** — quantizes the quantization constants themselves, saving an additional ~0.4 GB
3. **LoRA adapters trained in fp16** — so gradient quality is preserved even though base weights are quantized


## Step 2 — Config: model names, constants, seeds

In [11]:
# --- CELL 2: Config ---
import torch, random, numpy as np

# ── Model IDs ──────────────────────────────────────────────────────────────
# Qwen3.5 series (March 2026). No -Instruct suffix — all weights are instruct.
# Qwen3.5-9B is the closest public size to the 8B used in the InternAL paper.
MODEL_8B   = "Qwen/Qwen3.5-9B"   # replaces Qwen3-8B-Instruct
MODEL_17B  = "Qwen/Qwen3.5-2B"   # replaces Qwen3-1.7B-Instruct; used in Notebook C

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED_PRIMARY = 42
SEED_REPEAT  = 7   # used in E14-E17 seed repeats (Notebook C)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED_PRIMARY)

# ── MMLU 57 subjects ─────────────────────────────────────────────────────────
MMLU_SUBJECTS = [
    "abstract_algebra","anatomy","astronomy","business_ethics",
    "clinical_knowledge","college_biology","college_chemistry",
    "college_computer_science","college_mathematics","college_medicine",
    "college_physics","computer_security","conceptual_physics",
    "econometrics","electrical_engineering","elementary_mathematics",
    "formal_logic","global_facts","high_school_biology",
    "high_school_chemistry","high_school_computer_science",
    "high_school_european_history","high_school_geography",
    "high_school_government_and_politics","high_school_macroeconomics",
    "high_school_mathematics","high_school_microeconomics",
    "high_school_physics","high_school_psychology","high_school_statistics",
    "high_school_us_history","high_school_world_history","human_aging",
    "human_sexuality","international_law","jurisprudence","logical_fallacies",
    "machine_learning","management","marketing","medical_genetics",
    "miscellaneous","moral_disputes","moral_scenarios","nutrition",
    "philosophy","prehistory","professional_accounting","professional_law",
    "professional_medicine","professional_psychology","public_relations",
    "security_studies","sociology","us_foreign_policy","virology",
    "world_religions"
]

# ── Medical MMLU subjects (proximity group for H3) ──────────────────────────
# These 9 subjects are semantically proximate to medicine
MEDICAL_MMLU = {
    "anatomy", "clinical_knowledge", "college_biology", "college_medicine",
    "high_school_biology", "medical_genetics", "professional_medicine",
    "virology", "human_aging"
}

print(f"✅ Config set. CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ Config set. CUDA: True
   GPU: Tesla T4
   VRAM: 15.6 GB


---
## 🧪 The MMLU Benchmark — Why 57 Subjects?

**MMLU (Massive Multitask Language Understanding, Hendrycks et al., 2021)** is a 4-choice multiple-choice benchmark covering 57 academic subjects from elementary to professional level. It tests whether a model has acquired **broad, general world knowledge** during pre-training.

### Why MMLU is ideal for measuring forgetting

MMLU's breadth is exactly what makes it useful as a forgetting probe:

- **Coverage** — 57 subjects across STEM, humanities, social science, professional domains
- **Independence** — subjects are largely orthogonal, so forgetting in one does not contaminate another
- **Sensitivity** — small drops in accuracy (~2–5%) are statistically detectable across 57 subjects

### The medical proximity grouping

We partition the 57 subjects into:

| Group | Subjects | Rationale |
|-------|---------|-----------|
| **Proximal** (9) | anatomy, clinical\_knowledge, college\_biology, college\_medicine, high\_school\_biology, medical\_genetics, professional\_medicine, virology, human\_aging | Semantically close to MedQA (medicine, biology, physiology) |
| **Distal** (48) | All others | Semantically far from medicine |

If H3 is correct, fine-tuning on MedQA should cause **more forgetting** in proximal subjects than distal ones — because the weight updates that encode medical knowledge partially overlap with weights encoding related general knowledge.

### Seed reproducibility note

`SEED_PRIMARY = 42` fixes all stochastic elements: dataset sampling, model initialization (for eval), and training shuffles. This ensures results across E00–E04 are directly comparable.


## Step 3 — Utility functions (logger, eval, data)
These are shared across all experiments in this notebook.

In [12]:
# --- CELL 3: Result logger (saves immediately to Drive CSV) ---
import pandas as pd
from datetime import datetime

def log_result(exp_id: str, result_dict: dict):
    """
    Append one experiment row to the master CSV immediately.
    Call this after EVERY experiment — never batch results.
    """
    row = {
        "exp_id":    exp_id,
        "timestamp": datetime.now().isoformat(),
        **result_dict
    }
    df_new = pd.DataFrame([row])

    if os.path.exists(RESULTS_CSV):
        df_new.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_new.to_csv(RESULTS_CSV, index=False)

    print(f"  💾 Saved {exp_id} → {RESULTS_CSV}")
    return row

---
## 🔬 Evaluation Design — Chat Templates, Greedy Decoding, and the 5-Token Budget

### Why instruct models need chat templates

A base language model is trained to complete text. An *instruct* model (like Qwen3.5-9B) is additionally fine-tuned on `(instruction, response)` pairs wrapped in a **special token template** such as:

```
<|im_start|>user
{your prompt}<|im_end|>
<|im_start|>assistant
```

If you feed a raw prompt without this template, the model may produce garbled output because the probability distribution over the next token is conditioned on seeing these special tokens during training. `apply_chat_template()` handles this automatically.

### 🆕 Qwen3.5 thinking mode — why we disable it during evaluation

Qwen3.5 models support **extended thinking**: before answering, the model can emit a `<think>...</think>` block containing chain-of-thought reasoning. This is powerful for hard reasoning tasks, but **breaks MMLU evaluation**:

- The `<think>` block can be hundreds of tokens — `max_new_tokens=5` is exhausted before the answer letter is ever generated
- Even with a larger token budget, the regex `re.search(r"[ABCD]", ...)` would match letters *inside* the reasoning text rather than the final answer

We disable thinking mode with:
```python
tokenizer.apply_chat_template(..., enable_thinking=False)
```
This instructs the model to skip the `<think>` block and emit only the answer token directly.

**Fallback for older transformers builds**: if `enable_thinking` is not yet a recognised kwarg, we append ` /no_think` to the prompt — a special Qwen3.5 control token that achieves the same effect at the text level.

> ⚠️ **Training vs evaluation asymmetry**: `enable_thinking=False` is applied *only during eval*. During fine-tuning (`run_finetune`), thinking remains enabled so the model trains on its full instruction-following capability. Disabling thinking during SFT would artificially impoverish the training signal.

### Why greedy decoding (`do_sample=False`)?

For **evaluation**, we want *deterministic* outputs so results are reproducible across runs. Greedy decoding always picks the argmax token — no randomness, no temperature, no top-p. This is standard practice for multiple-choice benchmarks.

### Why `max_new_tokens=5`?

With thinking disabled, the model outputs only the answer letter directly. We cap at 5 tokens to:
1. Avoid wasting GPU time on any trailing explanation the model might still produce
2. Keep per-question latency minimal across 57 subjects × hundreds of questions

The regex `re.search(r"[ABCD]", gen_text)` extracts the first valid letter, making extraction robust to minor formatting variations (e.g. "Answer: B" → "B").


In [13]:
# --- CELL 4: MMLU evaluator ---
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import re

def eval_mmlu_subject(model, tokenizer, subject: str,
                      max_samples: int = None, verbose: bool = False) -> float:
    """
    Evaluate model on one MMLU subject. Returns accuracy (0.0–1.0).
    Uses chat template so Qwen instruct models respond correctly.
    """
    try:
        ds = load_dataset("cais/mmlu", subject, split="test", trust_remote_code=True)
    except Exception as e:
        print(f"  ⚠️  Could not load MMLU/{subject}: {e}")
        return float("nan")

    if max_samples:
        ds = ds.select(range(min(max_samples, len(ds))))

    correct, total = 0, 0
    label_map = {0: "A", 1: "B", 2: "C", 3: "D"}

    for ex in ds:
        choices_str = "\n".join(
            [f"{l}) {c}" for l, c in zip("ABCD", ex["choices"])]
        )
        prompt = (
            f"The following is a multiple choice question. "
            f"Answer with only the letter A, B, C, or D.\n\n"
            f"Question: {ex['question']}\n{choices_str}\n\nAnswer:"
        )

        # Use chat template — disable thinking so Qwen3.5 doesn't emit <think> blocks.
        # <think> tokens exhaust max_new_tokens=5 before the answer letter is generated.
        messages = [{"role": "user", "content": prompt}]
        try:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,   # CRITICAL: suppresses <think>…</think> for eval
            )
        except TypeError:
            # Fallback: older transformers builds don't support enable_thinking kwarg yet.
            # /no_think is Qwen3.5's text-level control token that has the same effect.
            messages_nothink = [{"role": "user", "content": prompt + " /no_think"}]
            try:
                text = tokenizer.apply_chat_template(
                    messages_nothink, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                text = prompt  # final fallback for non-chat base models

        ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

        with torch.no_grad():
            out = model.generate(
                ids,
                max_new_tokens=5,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Extract first A/B/C/D from generated tokens
        gen_text = tokenizer.decode(
            out[0, ids.shape[1]:], skip_special_tokens=True
        ).strip().upper()
        match = re.search(r"[ABCD]", gen_text)
        pred = match.group(0) if match else "X"

        gold = label_map[ex["answer"]]
        correct += int(pred == gold)
        total += 1

    acc = correct / total if total > 0 else float("nan")
    if verbose:
        print(f"    {subject}: {correct}/{total} = {acc:.3f}")
    return acc


def eval_all_mmlu(model, tokenizer, max_samples: int = None) -> dict:
    """
    Run eval on all 57 MMLU subjects.
    Returns dict: {subject_name: accuracy}
    Prints a progress bar.
    """
    results = {}
    n = len(MMLU_SUBJECTS)
    for i, subj in enumerate(MMLU_SUBJECTS):
        acc = eval_mmlu_subject(model, tokenizer, subj, max_samples=max_samples)
        results[subj] = acc
        # Simple progress print every 5 subjects
        if (i + 1) % 5 == 0 or (i + 1) == n:
            done = sum(1 for v in results.values() if not (isinstance(v, float) and v != v))
            print(f"  📊 [{i+1:2d}/{n}] last={subj} acc={acc:.3f}")
    return results

print("✅ Evaluator ready")

✅ Evaluator ready


---
## 📚 Training Data — MedQA-USMLE and Stratified Sampling

### Why MedQA?

**MedQA-USMLE** is a dataset of USMLE (United States Medical Licensing Examination) style 4-option multiple choice questions. It is:
- **Domain-specific** — purely medical, covering clinical reasoning, pharmacology, pathophysiology, anatomy, and more
- **High-quality** — questions are professionally authored, not scraped or synthesized
- **Challenging** — even GPT-4 achieves ~90%; smaller models score ~50–60%

Fine-tuning on MedQA is a controlled way to inject a narrow medical domain signal into the model while measuring what general knowledge is displaced.

### Why stratified sampling?

MedQA is tagged with `meta_info` categories (e.g. Step 1, Step 2 CK). If we sampled randomly, we might accidentally over-represent one category and under-represent another, introducing domain imbalance. **Stratified sampling** ensures each `meta_info` category contributes proportionally, giving us a cleaner "medicine in general" fine-tuning signal rather than "Step 2 clinical reasoning" specifically.

### SFT text format

Each training example is formatted as:
```
Question: <question text>
A) <choice>
B) <choice>
C) <choice>
D) <choice>
Answer: <correct letter>
```

This is a simple *supervised completion* format — the model learns to predict the answer letter given the question. No special tokens beyond what the tokenizer already knows.

### GSM8K and CodeAlpaca (used in Notebook B)

These loaders are defined here for reuse across notebooks:
- **GSM8K** — grade-school math word problems (control domain: numerics/reasoning)
- **CodeAlpaca** — programming instruction-response pairs (control domain: code)

They allow us to test whether forgetting patterns are specific to medicine or generalise across domains.


In [14]:
# --- CELL 5: MedQA stratified loader (as provided + safety checks) ---
from datasets import load_dataset, Dataset

def get_stratified_medqa(n_total: int = 4000, seed: int = 42) -> Dataset:
    """
    Load MedQA-USMLE-4-options train split, stratify by meta_info category.
    Formats each row as instruction-following text.
    """
    print("⬇️  Loading MedQA-USMLE-4-options ...")
    ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="train",
                      trust_remote_code=True)
    df = ds.to_pandas()
    print(f"   Raw dataset: {len(df)} rows, columns: {list(df.columns)}")

    # ── Detect column format ─────────────────────────────────────────────
    # Format A: option_0 ... option_3 (user's original code)
    # Format B: 'options' dict with keys opa/opb/opc/opd
    # Format C: 'options' is already a list
    def get_options(row):
        if "option_0" in df.columns:
            return [row[f"option_{i}"] for i in range(4)]
        elif "options" in df.columns:
            opts = row["options"]
            if isinstance(opts, dict):
                keys = sorted(opts.keys())[:4]
                return [opts[k] for k in keys]
            elif isinstance(opts, list):
                return opts[:4]
        return ["A", "B", "C", "D"]  # last resort

    # ── Stratified sample ────────────────────────────────────────────────
    if "meta_info" in df.columns and df["meta_info"].nunique() > 1:
        n_cats = df["meta_info"].nunique()
        n_per_cat = max(500, n_total // n_cats)
        sampled = (
            df.groupby("meta_info", group_keys=False)
              .apply(lambda x: x.sample(min(len(x), n_per_cat),
                                        random_state=seed))
        )
    else:
        sampled = df.sample(min(n_total, len(df)), random_state=seed)

    sampled = sampled.sample(frac=1, random_state=seed).reset_index(drop=True)
    print(f"   Stratified sample: {len(sampled)} rows")

    # ── Format as SFT text ───────────────────────────────────────────────
    rows = []
    for _, row in sampled.iterrows():
        opts = get_options(row)
        opts_str = "\n".join([f"{l}) {c}" for l, c in zip("ABCD", opts)])
        # answer is the letter (A/B/C/D)
        answer = str(row.get("answer", "A")).strip().upper()
        if answer not in "ABCD":
            answer = "A"
        rows.append({
            "text": (
                f"Question: {row['question']}\n"
                f"{opts_str}\n"
                f"Answer: {answer}"
            )
        })

    result_ds = Dataset.from_list(rows)
    print(f"✅ MedQA ready: {len(result_ds)} examples")
    return result_ds


def get_gsm8k(n_total: int = 4000, seed: int = 42) -> Dataset:
    """Load GSM8K for math domain control (E05)."""
    print("⬇️  Loading GSM8K ...")
    ds = load_dataset("gsm8k", "main", split="train", trust_remote_code=True)
    df = ds.to_pandas().sample(min(n_total, len(ds)), random_state=seed)
    rows = [{"text": f"Question: {r['question']}\nAnswer: {r['answer']}"}
            for _, r in df.iterrows()]
    result = Dataset.from_list(rows)
    print(f"✅ GSM8K ready: {len(result)} examples")
    return result


def get_code_alpaca(n_total: int = 4000, seed: int = 42) -> Dataset:
    """Load CodeAlpaca for code domain control (E05b)."""
    print("⬇️  Loading CodeAlpaca ...")
    ds = load_dataset("sahil2801/CodeAlpaca-20k", split="train",
                      trust_remote_code=True)
    df = ds.to_pandas().sample(min(n_total, len(ds)), random_state=seed)
    rows = [{"text": f"Instruction: {r['instruction']}\nResponse: {r['output']}"}
            for _, r in df.iterrows()]
    result = Dataset.from_list(rows)
    print(f"✅ CodeAlpaca ready: {len(result)} examples")
    return result

print("✅ Data loaders ready")

✅ Data loaders ready


---
## ⚙️ Fine-Tuning Setup — QLoRA + SFT Configuration Rationale

### NF4 quantization (`load_model_4bit`)

| Parameter | Value | Why |
|-----------|-------|-----|
| `load_in_4bit` | `True` | Compress weights to 4 bits → ~16 GB → ~5.5 GB |
| `bnb_4bit_quant_type` | `"nf4"` | Normal Float 4 is theoretically optimal for zero-mean Gaussian weight distributions |
| `bnb_4bit_compute_dtype` | `torch.float16` | Dequantise to fp16 for matrix multiplications → better accuracy than bf16 on T4 |
| `bnb_4bit_use_double_quant` | `True` | Quantise the 4-bit quantisation constants (saves ~0.4 GB extra) |

### LoRA configuration

| Parameter | Value | Why |
|-----------|-------|-----|
| `target_modules` | `q_proj, k_proj, v_proj, o_proj` | The four attention projection matrices; these govern how information is routed — most sensitive to domain shifts |
| `lora_alpha` | `rank * 2` | A common scaling convention; effective update magnitude ≈ $
rac{lpha}{r} = 2$ regardless of rank |
| `lora_dropout` | `0.05` | Mild regularisation; prevents the adapter from overfitting on 4 000 examples |
| `bias` | `"none"` | Don't train bias terms; keeps adapter small and avoids bias drift |

### Training hyperparameters (SFTConfig)

| Parameter | Value | Reasoning |
|-----------|-------|-----------|
| `max_steps` | 500 | Enough to saturate MedQA accuracy without over-training |
| `per_device_train_batch_size` | 2 | T4 constraint |
| `gradient_accumulation_steps` | 4 | Effective batch = 8; stabilises gradient estimates |
| `learning_rate` | 2e-4 | Standard for LoRA SFT; higher LRs cause instability with quantised base weights |
| `warmup_steps` | 50 | Prevents large early updates that could destabilise the frozen base |
| `fp16` | `True` | Train adapters in half precision; compatible with NF4 dequantised activations |
| `max_seq_length` | 512 | MedQA questions + answers are well within 512 tokens |

### Why we reload the base model fresh for each experiment

Each LoRA run calls `load_model_4bit` again, rather than reusing a cached model. This guarantees:
1. **No adapter bleed** — adapters from E01 don't contaminate E02
2. **Clean VRAM state** — we avoid fragmentation from previous training loops
3. **Reproducibility** — each fine-tuned model starts from an identical base checkpoint

### 🆕 No `enable_thinking` change in `run_finetune` — intentional

Qwen3.5's thinking mode is left **fully enabled during SFT**. The training examples are plain `text` strings (no chat template applied at training time by SFTTrainer by default). Thinking is a generation-time behaviour — it does not alter the gradient signal on the supervised completion targets in the SFT dataset. Disabling it during training would be both unnecessary and potentially harmful (it would inject `/no_think` tokens into every training prompt, distorting the fine-tuning distribution).


In [19]:
# --- CELL 6: LoRA fine-tuner (Kaggle-compatible, single-GPU, TRL 0.29) ---
import os, gc, ctypes, torch
from transformers import (BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

os.environ["CUDA_VISIBLE_DEVICES"]        = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]     = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_CUDA_WARMUP"] = "1"


def load_model_4bit(model_id: str):
    print(f"⬇️  Loading {model_id} in 4-bit NF4 ...")
    gc.collect()
    torch.cuda.empty_cache()
    ctypes.CDLL("libc.so.6").malloc_trim(0)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    free_gb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated(0)) / 1e9
    print(f"✅ Model loaded. VRAM free: {free_gb:.1f} GB")
    return model, tokenizer


def run_finetune(
    base_model_id: str,
    train_dataset,
    exp_id: str,
    lora_rank: int = 16,
    n_steps: int = 500,
    seed: int = 42,
) -> tuple:
    import os, gc, ctypes

    set_seed(seed)
    adapter_path = f"{DRIVE_BASE}/adapters/{exp_id}"
    os.makedirs(adapter_path, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"🔧 Fine-tuning: {exp_id}  rank={lora_rank}  steps={n_steps}")
    print(f"{'='*60}")

    for var in ['ft_model', 'ft_tok', 'model', 'tokenizer']:
        if var in globals():
            del globals()[var]
    gc.collect()
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    ctypes.CDLL("libc.so.6").malloc_trim(0)

    free_gb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated(0)) / 1e9
    print(f"   🧹 VRAM free before load: {free_gb:.1f} GB")

    model, tokenizer = load_model_4bit(base_model_id)

    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=lora_rank * 2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    # NOTE: enable_input_require_grads() intentionally removed —
    # it triggers the Qwen3.5 linear_attention shape bug with grad checkpointing
    model.print_trainable_parameters()

    sft_config = SFTConfig(
        output_dir=adapter_path,
        max_steps=n_steps,
        max_length=256,                    # ← reduced from 512 to save VRAM
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        save_steps=500,
        seed=seed,
        report_to="none",
        dataloader_pin_memory=False,
        gradient_checkpointing=False,      # ← OFF: incompatible with Qwen3.5 linear_attn
        ddp_find_unused_parameters=False,
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=sft_config,
    )

    print(f"🚀 Training for {n_steps} steps ...")
    trainer.train()

    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"💾 Adapter saved: {adapter_path}")

    return model, tokenizer


print("✅ Cell 6 ready")

✅ Cell 6 ready


---
## 📐 Semantic Proximity — Measuring How "Close" Each Subject Is to the Training Domain

### Hypothesis H1: Does forgetting correlate with semantic distance?

The **InternAL hypothesis** (H1) predicts that MMLU subjects whose content is semantically similar to the fine-tuning domain (MedQA) will suffer *more* forgetting, because:

- The weight subspaces encoding medical knowledge overlap with those encoding semantically adjacent general knowledge
- Gradient updates during medical fine-tuning partially overwrite these shared subspaces

We quantify semantic similarity using **cosine similarity in sentence embedding space**:

$$\text{sim}(\text{domain}, \text{subject}) = \frac{\mathbf{e}_{\text{domain}} \cdot \mathbf{e}_{\text{subject}}}{\|\mathbf{e}_{\text{domain}}\| \|\mathbf{e}_{\text{subject}}\|}$$

H1 is confirmed if **Pearson $r > 0$** between similarity scores and forgetting magnitudes across all 57 subjects.

### Why MiniLM?

`all-MiniLM-L6-v2` is a 6-layer, 22M-parameter sentence transformer fine-tuned on 1B sentence pairs for semantic similarity. It:
- Runs on **CPU in seconds** — no GPU overhead during evaluation
- Produces good sentence-level semantic similarity despite its small size
- Is widely used as a baseline embedding model in NLP research

### Hypothesis H3: Proximal vs distal t-test

H3 is a more focused version of H1: rather than correlating similarity with forgetting continuously, we divide subjects into two groups (proximal: `MEDICAL_MMLU`; distal: all others) and run a **Welch's t-test**. If proximal forgetting > distal forgetting with $p < 0.05$, H3 is supported.

The `direction` string makes the finding immediately interpretable:
- `"proximal>distal (matches InternAL)"` → H3 confirmed  
- `"proximal<=distal (reversal of InternAL)"` → H3 rejected


In [20]:
# --- CELL 7: Semantic proximity (MiniLM — CPU, fast) ---
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, ttest_ind
import numpy as np

# Domain description embeddings for H1 proximity test
DOMAIN_DESCRIPTIONS = {
    "medqa":      "medicine clinical knowledge anatomy pharmacology pathology diagnosis treatment disease symptoms",
    "gsm8k":      "mathematics arithmetic algebra word problems numerical computation",
    "codealpaca": "programming code software functions algorithms debugging python",
}

_embedder = None

def get_embedder():
    global _embedder
    if _embedder is None:
        print("⬇️  Loading MiniLM embedder (CPU) ...")
        _embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
        print("✅ MiniLM ready")
    return _embedder


def compute_subject_similarities(domain_key: str) -> dict:
    """
    Compute cosine similarity between domain description and each MMLU subject name.
    Returns dict: {subject: cosine_similarity}
    """
    emb = get_embedder()
    domain_desc = DOMAIN_DESCRIPTIONS[domain_key]
    domain_vec = emb.encode(domain_desc, normalize_embeddings=True)

    sims = {}
    subject_texts = [s.replace("_", " ") for s in MMLU_SUBJECTS]
    subject_vecs  = emb.encode(subject_texts, normalize_embeddings=True,
                               batch_size=32, show_progress_bar=False)

    for subj, vec in zip(MMLU_SUBJECTS, subject_vecs):
        sims[subj] = float(np.dot(domain_vec, vec))
    return sims


def run_proximity_test(
    domain_key: str,
    forgetting_dict: dict,
    exp_id: str,
) -> dict:
    """
    H1 test: Pearson r between cosine similarity and forgetting per subject.
    H3 test: two-sample t-test, proximal subjects vs distal subjects.
    Returns dict with all stats.
    """
    sims = compute_subject_similarities(domain_key)

    # Filter to subjects where we have valid forgetting values
    valid_subjects = [s for s in MMLU_SUBJECTS
                      if s in forgetting_dict
                      and not np.isnan(forgetting_dict[s])]

    sim_vals = [sims[s] for s in valid_subjects]
    fgt_vals = [forgetting_dict[s] for s in valid_subjects]

    # ── H1: Pearson correlation ──────────────────────────────────────────
    r, p = pearsonr(sim_vals, fgt_vals)

    # ── H3: Proximal vs distal t-test (direction test) ──────────────────
    proximal_fgt = [forgetting_dict[s] for s in valid_subjects
                    if s in MEDICAL_MMLU]
    distal_fgt   = [forgetting_dict[s] for s in valid_subjects
                    if s not in MEDICAL_MMLU]

    if len(proximal_fgt) > 1 and len(distal_fgt) > 1:
        t_stat, t_p = ttest_ind(proximal_fgt, distal_fgt)
        proximal_mean = np.mean(proximal_fgt)
        distal_mean   = np.mean(distal_fgt)
        # H3 direction: if proximal_mean < distal_mean (more forgetting nearby)
        # → matches InternAL. If proximal_mean >= distal_mean → reversal.
        direction = "proximal>distal (matches InternAL)" \
                    if proximal_mean < distal_mean \
                    else "proximal<=distal (reversal of InternAL)"
    else:
        t_stat, t_p = float("nan"), float("nan")
        proximal_mean, distal_mean = float("nan"), float("nan")
        direction = "insufficient data"

    result = {
        "domain": domain_key,
        "h1_pearson_r": round(r, 4),
        "h1_pearson_p": round(p, 6),
        "h3_t_stat":    round(t_stat, 4) if not np.isnan(t_stat) else "nan",
        "h3_t_p":       round(t_p, 6)    if not np.isnan(t_p)    else "nan",
        "h3_proximal_mean_forgetting": round(proximal_mean, 4) if not np.isnan(proximal_mean) else "nan",
        "h3_distal_mean_forgetting":   round(distal_mean, 4)   if not np.isnan(distal_mean)   else "nan",
        "h3_direction": direction,
        "n_subjects": len(valid_subjects),
    }

    print(f"\n📐 Proximity test [{exp_id}] domain={domain_key}")
    print(f"   H1 Pearson r = {r:.3f}  p = {p:.5f}")
    print(f"   H3 proximal_mean = {proximal_mean:.3f}  distal_mean = {distal_mean:.3f}")
    print(f"   H3 direction: {direction}")
    print(f"   H3 t = {t_stat:.3f}  p = {t_p:.5f}")

    return result

print("✅ Proximity tester ready")

✅ Proximity tester ready


---
## EXPERIMENT E00 — Baseline (all 57 MMLU subjects, no fine-tuning)
⏱️ ~2 hours on free T4  
**This must finish before any other experiment.**

### Why do we need a baseline?

Forgetting is defined as the **change** in accuracy:

$$\text{forgetting}_{\text{subj}} = \text{acc}_{\text{post}} - \text{acc}_{\text{pre}}$$

A negative value means the model got worse on that subject after fine-tuning. Without `acc_pre` (the baseline), we cannot compute this quantity.

The baseline also tells us:
- **Which subjects the model is already good/bad at** — high-variance subjects (where the model scores ~25%, near chance) are noisier and contribute less signal to the forgetting analysis
- **Whether the model has any medical knowledge pre-fine-tuning** — the medical MMLU subjects should score above chance even before MedQA fine-tuning, since Qwen3-8B was pre-trained on a large web corpus

### Crash-resilient checkpointing

E00 takes ~2 hours. Free T4 sessions disconnect after ~12 hours, but occasionally crash earlier. The checkpoint mechanism:

1. Every 5 subjects: saves completed results to `E00_checkpoint.csv`
2. On re-run: loads the checkpoint and **skips already-completed subjects**
3. This means a crash costs at most 5 subjects (~10 minutes) of re-computation

Always re-run from the top of the notebook after a crash — the Drive mount and config cells must execute before E00 can resume.


In [16]:
# ── CELL: Force VRAM reset before loading ──
import torch, gc

# Delete any leftover model references
for var in ['model_base', 'tok_base', 'ft_model', 'ft_tok', 'model', 'tokenizer']:
    if var in dir():
        del globals()[var]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Confirm free memory
t = torch.cuda.get_device_properties(0)
free = t.total_memory - torch.cuda.memory_allocated(0)
print(f"GPU: {t.name}")
print(f"Total : {t.total_memory/1e9:.2f} GB")
print(f"In use: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"Free  : {free/1e9:.2f} GB")

GPU: Tesla T4
Total : 15.64 GB
In use: 0.00 GB
Free  : 15.64 GB


In [17]:
# --- CELL 8: E00 — Load model ---
# Load model ONCE and reuse for all 57 subject evaluations
print("\n" + "="*60)
print("EXPERIMENT E00 — BASELINE MMLU EVALUATION")
print("="*60)

model_base, tok_base = load_model_4bit(MODEL_8B)
model_base.eval()



EXPERIMENT E00 — BASELINE MMLU EVALUATION
⬇️  Loading Qwen/Qwen3.5-9B in 4-bit NF4 ...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 4096)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (in_proj_qkv): Linear4bit(in_features=4096, out_features=8192, bias=False)
          (in_proj_z): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (in_proj_b): Linear4bit(in_features=4096, out_features=32, bias=False)
          (in_proj_a): Linear4bit(in_features=4096, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)


In [18]:
# --- CELL 9: E00 — Run all 57 subjects ---
# Results are saved after every 5 subjects as a safety checkpoint.
set_seed(SEED_PRIMARY)

print("🔍 Evaluating all 57 MMLU subjects on baseline model ...")
print("   (Saves checkpoint every 5 subjects)")

baseline_accs = {}
checkpoint_path = f"/kaggle/input/datasets/akankshanarula/lora-results/E00_checkpoint.csv"

# Check if we crashed mid-run and can resume
if os.path.exists(checkpoint_path):
    existing = pd.read_csv(checkpoint_path)
    baseline_accs = dict(zip(existing["subject"], existing["accuracy"]))
    print(f"   ♻️  Resuming from checkpoint: {len(baseline_accs)} subjects done")

for i, subj in enumerate(MMLU_SUBJECTS):
    if subj in baseline_accs:
        print(f"   ⏭️  Skipping {subj} (already done: {baseline_accs[subj]:.3f})")
        continue

    acc = eval_mmlu_subject(model_base, tok_base, subj, verbose=True)
    baseline_accs[subj] = acc

    # Save checkpoint every 5 subjects
    if (i + 1) % 5 == 0 or (i + 1) == len(MMLU_SUBJECTS):
        ckpt_df = pd.DataFrame([
            {"subject": s, "accuracy": a}
            for s, a in baseline_accs.items()
        ])
        ckpt_df.to_csv(checkpoint_path, index=False)
        print(f"  💾 Checkpoint saved [{i+1}/57]")

# Save final baseline CSV
baseline_df = pd.DataFrame([
    {"subject": s, "accuracy": a, "is_medical": s in MEDICAL_MMLU}
    for s, a in baseline_accs.items()
])
baseline_df.to_csv(BASELINE_CSV, index=False)

# Log to master results
for subj, acc in baseline_accs.items():
    log_result("E00", {
        "model": MODEL_8B,
        "domain": "none",
        "lora_rank": 0,
        "n_steps": 0,
        "replay_size": 0,
        "seed": SEED_PRIMARY,
        "subject": subj,
        "accuracy": round(acc, 4),
        "is_medical": subj in MEDICAL_MMLU,
        "delta_acc": 0.0,   # baseline has no delta
        "forgetting": 0.0,
    })

print("\n✅ E00 COMPLETE")
print(f"   Mean MMLU accuracy: {np.nanmean(list(baseline_accs.values())):.3f}")
print(f"   Medical subjects mean: {np.nanmean([baseline_accs[s] for s in MEDICAL_MMLU if s in baseline_accs]):.3f}")
print(f"   Non-medical mean: {np.nanmean([v for s,v in baseline_accs.items() if s not in MEDICAL_MMLU]):.3f}")
print(f"   Saved: {BASELINE_CSV}")

🔍 Evaluating all 57 MMLU subjects on baseline model ...
   (Saves checkpoint every 5 subjects)
   ♻️  Resuming from checkpoint: 57 subjects done
   ⏭️  Skipping abstract_algebra (already done: 0.570)
   ⏭️  Skipping anatomy (already done: 0.800)
   ⏭️  Skipping astronomy (already done: 0.895)
   ⏭️  Skipping business_ethics (already done: 0.810)
   ⏭️  Skipping clinical_knowledge (already done: 0.830)
   ⏭️  Skipping college_biology (already done: 0.910)
   ⏭️  Skipping college_chemistry (already done: 0.570)
   ⏭️  Skipping college_computer_science (already done: 0.700)
   ⏭️  Skipping college_mathematics (already done: 0.550)
   ⏭️  Skipping college_medicine (already done: 0.775)
   ⏭️  Skipping college_physics (already done: 0.647)
   ⏭️  Skipping computer_security (already done: 0.810)
   ⏭️  Skipping conceptual_physics (already done: 0.838)
   ⏭️  Skipping econometrics (already done: 0.675)
   ⏭️  Skipping electrical_engineering (already done: 0.807)
   ⏭️  Skipping elementary_mat

---
## EXPERIMENTS E01–E04 — Rank Ablation (H2)
Fine-tune on MedQA at r=4, 16, 64, 128. Domain held constant.  
Tests **Biderman 2024 vs Steele 2026 (Contradiction #2)**.  
⏱️ ~11 GPU-hours total across 4 runs

### H2 — Does LoRA Rank Control Forgetting Severity?

This is the central contradiction motivating this notebook:

| Claim | Source | Prediction |
|-------|--------|-----------|
| Higher rank → more forgetting | Biderman et al., 2024 | Mean MMLU drop should increase monotonically: r4 < r16 < r64 < r128 |
| Rank has no effect on forgetting | Steele, 2026 | Mean MMLU drop should be flat across r4, r16, r64, r128 |

**Why might higher rank cause more forgetting?**

A higher-rank adapter has more trainable parameters, so it has greater capacity to reshape the model's weight matrices. In principle, this means it can push weight subspaces further from their pre-trained positions — overwriting more general knowledge in the process.

**Why might rank have no effect?**

LoRA updates are always constrained to a low-dimensional subspace relative to the full weight matrix. Even at r=128, the adapter only has ~0.1–0.5% of the full model's parameters. The *learning rate* and *number of steps* may dominate over rank in determining forgetting magnitude.

**Experimental design**

- Rank is the **sole variable**: r ∈ {4, 16, 64, 128}
- Domain (MedQA), steps (500), LR (2e-4), seed (42) are held constant
- After each fine-tuning run, **all 57 MMLU subjects** are re-evaluated
- `forgetting[subj] = post_acc[subj] - baseline_accs[subj]` (negative = forgetting)
- H1 and H3 proximity tests run automatically after each experiment

**What to look for in the output summary table**

```
Exp    Rank   Mean fgt    Med fgt     H1 r
----------------------------------------------
E01    4      ±X.XXX      ±X.XXX      X.XXX
E02    16     ±X.XXX      ±X.XXX      X.XXX
E03    64     ±X.XXX      ±X.XXX      X.XXX
E04    128    ±X.XXX      ±X.XXX      X.XXX
```

- **Monotonic increase** in `Mean fgt` → supports Biderman
- **Flat or non-monotonic** → supports Steele
- **H1 r > 0 consistently** → supports InternAL proximity hypothesis


In [21]:
# --- CELL 10: Load training data once ---
medqa_train = get_stratified_medqa(n_total=4000, seed=SEED_PRIMARY)
print(f"\n✅ MedQA training data loaded: {len(medqa_train)} examples")
print(f"   Sample: {medqa_train[0]['text'][:200]} ...")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'GBaker/MedQA-USMLE-4-options' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


⬇️  Loading MedQA-USMLE-4-options ...
   Raw dataset: 10178 rows, columns: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases']
   Stratified sample: 4000 rows


/tmp/ipykernel_55/1676844107.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), n_per_cat),


✅ MedQA ready: 4000 examples

✅ MedQA training data loaded: 4000 examples
   Sample: Question: A 52-year-old female with a history of poorly-controlled diabetes presents to her primary care physician because of pain and tingling in her hands. These symptoms began several months ago an ...


In [22]:

# --- CELL 11: Rank ablation loop ---
# E01=r4, E02=r16 (primary), E03=r64, E04=r128
# Each run: fine-tune → re-eval 57 subjects → compute forgetting → save

RANK_EXPERIMENTS = [
    ("E01", 4,   500),
    ("E02", 16,  500),   # primary condition
    ("E03", 64,  500),
    ("E04", 128, 500),
]

rank_results = {}   # {exp_id: {subject: acc_after}}

for exp_id, rank, n_steps in RANK_EXPERIMENTS:
    print(f"\n{'='*60}")
    print(f"▶  {exp_id}: rank={rank}, steps={n_steps}")
    print(f"{'='*60}")

    # ── Fine-tune ────────────────────────────────────────────────────────
    ft_model, ft_tok = run_finetune(
        base_model_id=MODEL_8B,
        train_dataset=medqa_train,
        exp_id=exp_id,
        lora_rank=rank,
        n_steps=n_steps,
        seed=SEED_PRIMARY,
    )
    ft_model.eval()

    # ── Re-evaluate all 57 MMLU subjects ────────────────────────────────
    print(f"\n📊 Re-evaluating MMLU after {exp_id} ...")
    post_accs = eval_all_mmlu(ft_model, ft_tok)
    rank_results[exp_id] = post_accs

    # ── Compute forgetting and log ───────────────────────────────────────
    forgetting = {}
    for subj in MMLU_SUBJECTS:
        base_acc = baseline_accs.get(subj, float("nan"))
        post_acc = post_accs.get(subj, float("nan"))
        delta    = post_acc - base_acc   # negative = forgetting
        forgetting[subj] = delta

        log_result(exp_id, {
            "model":      MODEL_8B,
            "domain":     "medqa",
            "lora_rank":  rank,
            "n_steps":    n_steps,
            "replay_size": 0,
            "seed":       SEED_PRIMARY,
            "subject":    subj,
            "accuracy":   round(post_acc, 4) if not np.isnan(post_acc) else "nan",
            "is_medical": subj in MEDICAL_MMLU,
            "delta_acc":  round(delta, 4)    if not np.isnan(delta)    else "nan",
            "forgetting": round(-delta, 4)   if not np.isnan(delta)    else "nan",
        })

    # ── H1+H3 proximity test ─────────────────────────────────────────────
    prox_stats = run_proximity_test("medqa", forgetting, exp_id)
    log_result(f"{exp_id}_proximity", {
        "model": MODEL_8B, "domain": "medqa", "lora_rank": rank,
        "n_steps": n_steps, "replay_size": 0, "seed": SEED_PRIMARY,
        "subject": "ALL_PROXIMITY_STATS",
        **prox_stats,
        "delta_acc": 0, "forgetting": 0, "accuracy": 0,
        "is_medical": False,
    })

    # ── Summary print ─────────────────────────────────────────────────────
    mean_fgt = -np.nanmean(list(forgetting.values()))
    med_fgt  = -np.nanmean([forgetting[s] for s in MEDICAL_MMLU if s in forgetting])
    gen_fgt  = -np.nanmean([v for s,v in forgetting.items() if s not in MEDICAL_MMLU])

    print(f"\n📋 {exp_id} SUMMARY (rank={rank})")
    print(f"   Mean forgetting:          {mean_fgt:+.3f}")
    print(f"   Medical subjects fgt:     {med_fgt:+.3f}")
    print(f"   Non-medical subjects fgt: {gen_fgt:+.3f}")
    print(f"   H1 Pearson r:             {prox_stats['h1_pearson_r']:.3f}")
    print(f"   H3 direction:             {prox_stats['h3_direction']}")

    # ── Free VRAM before next run ────────────────────────────────────────
    del ft_model, ft_tok
    torch.cuda.empty_cache()
    import gc; gc.collect()
    print(f"   🗑️  VRAM cleared")

print("\n" + "="*60)
print("✅ ALL RANK ABLATION EXPERIMENTS COMPLETE (E01–E04)")
print("="*60)

# ── Quick H2 summary table ───────────────────────────────────────────────────
print("\n📊 H2 — Rank effect summary:")
print(f"{'Exp':6} {'Rank':6} {'Mean fgt':10} {'Med fgt':10} {'H1 r':8}")
print("-"*44)
df_all = pd.read_csv(RESULTS_CSV)
for exp_id, rank, _ in RANK_EXPERIMENTS:
    df_e = df_all[df_all["exp_id"] == exp_id]
    if len(df_e) == 0:
        continue
    df_e = df_e[pd.to_numeric(df_e["forgetting"], errors="coerce").notna()]
    df_e["forgetting_num"] = pd.to_numeric(df_e["forgetting"])
    mean_fgt = df_e["forgetting_num"].mean()
    med_fgt  = df_e[df_e["is_medical"] == True]["forgetting_num"].mean()
    # get proximity r
    df_prox = df_all[df_all["exp_id"] == f"{exp_id}_proximity"]
    r_val = df_prox["h1_pearson_r"].values[0] if len(df_prox) > 0 else "n/a"
    print(f"{exp_id:6} {rank:6} {mean_fgt:+10.3f} {med_fgt:+10.3f} {str(r_val):8}")

print(f"\n✅ Results saved to: {RESULTS_CSV}")
print("   Hand this file to the analysis notebook (Notebook D) when done.")


▶  E01: rank=4, steps=500

🔧 Fine-tuning: E01  rank=4  steps=500
   🧹 VRAM free before load: 8.0 GB
⬇️  Loading Qwen/Qwen3.5-9B in 4-bit NF4 ...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

✅ Model loaded. VRAM free: 0.3 GB
trainable params: 983,040 || all params: 8,954,786,304 || trainable%: 0.0110


Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


🚀 Training for 500 steps ...


ValueError: too many values to unpack (expected 3)

In [ ]:
# --- CELL 12: Cleanup ---
del model_base, tok_base
torch.cuda.empty_cache()
import gc; gc.collect()
print("✅ Notebook A complete. GPU memory cleared.")
print(f"   All results in: {RESULTS_CSV}")
print(f"   All adapters in: {DRIVE_BASE}/adapters/")